# Temporal Reasoning with OWL Time: Site Periods Relative to the *Clades Variana* (AD 9)

**Florian Thiery · LEIZA · CAA 2026**

---

This notebook demonstrates two complementary approaches to identifying archaeological site periods that occurred *after or during* the *Clades Variana* (Battle of the Teutoburg Forest, AD 9), using a Linked Open Data graph of Arretine ware find sites.

The dataset (`arretine_sites_minigraph.ttl`) encodes temporal extents via [OWL Time](https://www.w3.org/TR/owl-time/) (`time:Interval`, `time:hasBeginning`, `time:hasEnd`, `time:inXSDgYear`) and [Allen's interval algebra](https://dl.acm.org/doi/10.1145/182.358434) relations between period clusters.

| Approach | Method | Granularity |
|---|---|---|
| **A — Absolute** | Numeric comparison of `xsd:gYear` values | Site level |
| **B — Relative** | Allen relations via cluster membership | Cluster → Site level |

> **Note on `xsd:gYear` casting:** rdflib's SPARQL engine does not support `xsd:integer(?gYear)` casts inline. Year values are therefore retrieved as strings and converted to integers in Python.

## 1 · Setup

In [1]:
import warnings, logging
warnings.filterwarnings('ignore')
# rdflib on older Python/isodate versions logs xsd:gYear parse failures
# to stderr via Python's logging module — suppress at root level
logging.disable(logging.CRITICAL)

from pathlib import Path
import json as _json
import pandas as pd
from rdflib import Graph
from IPython.display import IFrame

TTL_PATH = Path('arretine_sites_minigraph.ttl')  # adjust path if needed

g = Graph()
g.parse(TTL_PATH, format='turtle')

# Re-enable logging after graph load so later cells aren't silenced
logging.disable(logging.NOTSET)

print(f'Graph loaded: {len(g):,} triples from "{TTL_PATH.name}"')


Graph loaded: 1,830 triples from "arretine_sites_minigraph.ttl"


## 2 · Helper Functions

In [2]:
def gyear_to_int(literal) -> int:
    """
    Convert an xsd:gYear RDF literal to a Python integer.

    xsd:gYear uses the proleptic Gregorian calendar in astronomical
    year numbering: '0009' = AD 9, '-0008' = 9 BC (there is no year 0
    in the historical calendar, but xsd:gYear includes it).
    """
    return int(str(literal))


def allen_relation(a0: int, a1: int, b0: int, b1: int) -> str:
    """
    Return the Allen interval relation of interval A = [a0, a1]
    with respect to interval B = [b0, b1].

    Implements all 13 basic relations from Allen (1983).
    """
    if   a1 < b0:                       return "before"
    elif a0 > b1:                       return "after"
    elif a1 == b0:                      return "meets"
    elif a0 == b1:                      return "metBy"
    elif a0 == b0 and a1 == b1:         return "equals"
    elif a0 < b0  and a1 > b1:          return "contains"
    elif a0 > b0  and a1 < b1:          return "during"
    elif a0 == b0 and a1 < b1:          return "starts"
    elif a0 == b0 and a1 > b1:          return "startedBy"
    elif a0 < b0  and a1 == b1:         return "finishedBy"
    elif a0 > b0  and a1 == b1:         return "finishes"
    elif a0 < b0  and a1 < b1:          return "overlaps"
    elif a0 > b0  and a1 > b1:          return "overlappedBy"
    return "unknown"


# Relations representing 'after or concurrent with' the reference event.
# 'meets' is included: a site period ending exactly at AD 9
# was demonstrably active during the event (its final year coincides).
CONTEMPORARY_OR_LATER = {
    "after",        # site begins after event ends
    "metBy",        # site begins exactly where event ends
    "equals",       # identical temporal extent
    "during",       # site falls entirely within event span
    "starts",       # site begins with event, ends earlier
    "startedBy",    # site begins with event, ends later
    "finishes",     # site ends with event, begins later
    "finishedBy",   # site ends with event, begins earlier
    "overlappedBy", # site begins before event ends, ends after
    "contains",     # site spans the entire event
    "meets",        # site ends exactly at event start (active in AD 9)
}

## 3 · Reference Event: *Clades Variana*

In [3]:
q_event = """
    PREFIX ae:   <http://leiza-scit.github.io/CAA2026-alligator/>
    PREFIX time: <http://www.w3.org/2006/time#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?label ?begin ?end
    WHERE {
        ae:event_Clades_Variana
            rdfs:label                        ?label ;
            time:hasBeginning/time:inXSDgYear ?begin ;
            time:hasEnd/time:inXSDgYear       ?end .
    }
"""

ev = list(g.query(q_event))[0]
EV_LABEL = str(ev[0])
EV_BEGIN = gyear_to_int(ev[1])
EV_END   = gyear_to_int(ev[2])

print(f"Reference event : {EV_LABEL}")
print(f"Temporal extent : AD {EV_BEGIN}\u2013AD {EV_END}  (point event)")

Reference event : Clades Variana
Temporal extent : AD 9–AD 9  (point event)


## 4 · Approach A: Absolute Temporal Comparison

All sites are retrieved via SPARQL. For each site, the Allen relation of its temporal
interval `[start, end]` to the *Clades Variana* interval `[9, 9]` is computed in Python.
Sites whose relation belongs to `CONTEMPORARY_OR_LATER` are retained.

In [4]:
q_sites = """
    PREFIX time: <http://www.w3.org/2006/time#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX fsl:  <http://fuzzy-sl.squirrel.link/ontology/>

    SELECT ?site ?label ?startYear ?endYear
    WHERE {
        ?site a time:Interval ;
              a fsl:Site ;
              rdfs:label                        ?label ;
              time:hasBeginning/time:inXSDgYear ?startYear ;
              time:hasEnd/time:inXSDgYear       ?endYear .
    }
    ORDER BY ?startYear ?label
"""

rows = []
for r in g.query(q_sites):
    s = gyear_to_int(r[2])
    e = gyear_to_int(r[3])
    rel = allen_relation(s, e, EV_BEGIN, EV_END)
    rows.append({
        "site_id":   str(r[0]).split("/")[-1],
        "label":     str(r[1]),
        "start":     s,
        "end":       e,
        "allen_rel": rel,
    })

df_all = pd.DataFrame(rows)
df_A = (
    df_all[df_all["allen_rel"].isin(CONTEMPORARY_OR_LATER)]
    .sort_values(["allen_rel", "start", "label"])
    .reset_index(drop=True)
)

print(f"Sites contemporary with or later than {EV_LABEL}: {len(df_A)} of {len(df_all)}")
print(f"\nRelation breakdown:")
print(df_A["allen_rel"].value_counts().to_string())
df_A

Sites contemporary with or later than Clades Variana: 32 of 44

Relation breakdown:
allen_rel
meets       15
contains    11
after        6


,site_id,label,start,end,allen_rel
0,vJnM0D,"Augst, Insula 20",16,28,after
1,p4zxM0,"Augst, Theater",16,28,after
2,JmWOrl,Maastricht,16,28,after
3,KmaGQd,Vechten,16,28,after
4,EOD041,Velsen,16,28,after
5,xbEdvL,"Zurzach, Lager",16,28,after
6,Qz2oRe,Worms,-16,13,contains
7,zggRNd,"Augsburg, Stadt",7,13,contains
8,rR3Mgy,"Augst, Insula 31",7,13,contains
9,rY6gGz,"Avenches, Insula 15",7,13,contains


## 5 · Approach B: Relative Reasoning via Allen Relations on Cluster Level

The TTL dataset encodes explicit `time:interval*` relations between period *clusters*,
not between individual sites and events. We therefore:
1. Retrieve all clusters with their temporal extents and site memberships.
2. Compute the Allen relation of each cluster to the *Clades Variana* in Python.
3. Retain clusters (and their member sites) where the relation is in `CONTEMPORARY_OR_LATER`.

In [5]:
q_clusters = """
    PREFIX time: <http://www.w3.org/2006/time#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX lado: <http://archaeology.link/ontology#>

    SELECT ?cluster ?clusterLabel ?cBegin ?cEnd ?site ?siteLabel ?sBegin ?sEnd
    WHERE {
        ?cluster a lado:PeriodCluster ;
                 rdfs:label                        ?clusterLabel ;
                 time:hasBeginning/time:inXSDgYear ?cBegin ;
                 time:hasEnd/time:inXSDgYear       ?cEnd ;
                 lado:hasClusterMember             ?site .

        ?site rdfs:label                        ?siteLabel ;
              time:hasBeginning/time:inXSDgYear ?sBegin ;
              time:hasEnd/time:inXSDgYear       ?sEnd .
    }
    ORDER BY ?clusterLabel ?siteLabel
"""

rows_b = []
for r in g.query(q_clusters):
    cb = gyear_to_int(r[2])
    ce = gyear_to_int(r[3])
    sb = gyear_to_int(r[6])
    se = gyear_to_int(r[7])
    cluster_rel = allen_relation(cb, ce, EV_BEGIN, EV_END)

    if cluster_rel in CONTEMPORARY_OR_LATER:
        rows_b.append({
            "cluster":     str(r[1]),
            "c_start":     cb,
            "c_end":       ce,
            "cluster_rel": cluster_rel,
            "site":        str(r[5]),
            "site_start":  sb,
            "site_end":    se,
        })

df_B = (
    pd.DataFrame(rows_b)
    .sort_values(["cluster_rel", "c_start", "site"])
    .reset_index(drop=True)
)

print(f"Qualifying sites (via cluster membership): {df_B['site'].nunique()} "
      f"across {df_B['cluster'].nunique()} clusters")
df_B

Qualifying sites (via cluster membership): 32 across 5 clusters


,cluster,c_start,c_end,cluster_rel,site,site_start,site_end
0,Period cluster 16 – 28 CE,16,28,after,"Augst, Insula 20",16,28
1,Period cluster 16 – 28 CE,16,28,after,"Augst, Theater",16,28
2,Period cluster 16 – 28 CE,16,28,after,Maastricht,16,28
3,Period cluster 16 – 28 CE,16,28,after,Vechten,16,28
4,Period cluster 16 – 28 CE,16,28,after,Velsen,16,28
5,Period cluster 16 – 28 CE,16,28,after,"Zurzach, Lager",16,28
6,Period cluster 16 BCE – 13 CE,-16,13,contains,Worms,-16,13
7,Period cluster 7 – 13 CE,7,13,contains,"Augsburg, Stadt",7,13
8,Period cluster 7 – 13 CE,7,13,contains,"Augst, Insula 31",7,13
9,Period cluster 7 – 13 CE,7,13,contains,"Avenches, Insula 15",7,13


## 6 · Comparison: Approaches A and B

Both approaches should return identical site sets if the cluster membership
is consistent with the absolute date ranges in the data.

In [6]:
sites_A = set(df_A["label"])
sites_B = set(df_B["site"])

print(f"Approach A (absolute) : {len(sites_A):>3} sites")
print(f"Approach B (relative) : {len(sites_B):>3} sites")
print(f"Intersection          : {len(sites_A & sites_B):>3} sites")

only_A = sites_A - sites_B
only_B = sites_B - sites_A
if only_A:
    print(f"\nOnly in A (not clustered): {sorted(only_A)}")
if only_B:
    print(f"\nOnly in B (cluster mis-match): {sorted(only_B)}")
if not only_A and not only_B:
    print("\n\u2713  Both approaches return identical site sets.")

Approach A (absolute) :  32 sites
Approach B (relative) :  32 sites
Intersection          :  32 sites

✓  Both approaches return identical site sets.


## 7 · Cluster Overview — Allen Relations to the *Clades Variana*

In [7]:
q_all_clusters = """
    PREFIX time: <http://www.w3.org/2006/time#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX lado: <http://archaeology.link/ontology#>

    SELECT ?clusterLabel ?cBegin ?cEnd (COUNT(?site) AS ?nSites)
    WHERE {
        ?cluster a lado:PeriodCluster ;
                 rdfs:label                        ?clusterLabel ;
                 time:hasBeginning/time:inXSDgYear ?cBegin ;
                 time:hasEnd/time:inXSDgYear       ?cEnd ;
                 lado:hasClusterMember             ?site .
    }
    GROUP BY ?clusterLabel ?cBegin ?cEnd
    ORDER BY ?cBegin
"""

rows_c = []
for r in g.query(q_all_clusters):
    cb = gyear_to_int(r[1])
    ce = gyear_to_int(r[2])
    rel = allen_relation(cb, ce, EV_BEGIN, EV_END)
    rows_c.append({
        "cluster":        str(r[0]),
        "start (CE/BCE)": cb,
        "end (CE/BCE)":   ce,
        "allen_rel":      rel,
        "sites":          int(str(r[3])),
        "included":       "\u2713" if rel in CONTEMPORARY_OR_LATER else "\u2717",
    })

pd.DataFrame(rows_c).sort_values("start (CE/BCE)")

,cluster,start (CE/BCE),end (CE/BCE),allen_rel,sites,included
7,Period cluster 16 BCE – 8 CE,-16,8,before,4,✗
6,Period cluster 16 BCE – 9 CE,-16,9,meets,3,✓
5,Period cluster 16 BCE – 9 BCE,-16,-9,before,2,✗
4,Period cluster 16 BCE – 13 CE,-16,13,contains,1,✓
3,Period cluster 15 BCE – 9 BCE,-15,-9,before,3,✗
2,Period cluster 12 BCE – 8 CE,-12,8,before,2,✗
1,Period cluster 11 BCE – 8 CE,-11,8,before,1,✗
0,Period cluster 7 BCE – 9 CE,-7,9,meets,12,✓
8,Period cluster 7 – 13 CE,7,13,contains,10,✓
9,Period cluster 16 – 28 CE,16,28,after,6,✓


## 8 · Visualisation A: Distribution of Allen Relations Across All Sites

This chart illustrates how the full site corpus of 44 sites distributes across
all 13 Allen relations relative to the *Clades Variana*.
Highlighted bars (blue) represent the qualifying relations used in Approach A.

In [8]:
import uuid, json as _j
from pathlib import Path
from IPython.display import IFrame

ALL_RELS = ['before','meets','overlaps','finishedBy','contains',
            'starts','equals','startedBy','during','finishes',
            'overlappedBy','metBy','after']
rel_counts = df_all['allen_rel'].value_counts().to_dict()
counts  = [rel_counts.get(r, 0) for r in ALL_RELS]
colours = ['#185FA5' if r in CONTEMPORARY_OR_LATER else '#B4B2A9' for r in ALL_RELS]
n_sites = len(df_all)
ch = max(340, len(ALL_RELS) * 30 + 80)
cid = 'c' + uuid.uuid4().hex

chart_html = '''<!DOCTYPE html>
<html><head><meta charset="utf-8">
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>
</head><body style="margin:0;font-family:sans-serif;padding:8px 4px 4px">
<p style="font-size:12px;color:#555;margin:0 0 6px">
  All ''' + str(n_sites) + ''' sites
  <span style="display:inline-block;width:10px;height:10px;background:#185FA5;
    border-radius:2px;vertical-align:middle"></span>
  Contemporary or later than <em>Clades Variana</em> (AD&nbsp;9)
  <span style="display:inline-block;width:10px;height:10px;background:#B4B2A9;
    border-radius:2px;vertical-align:middle"></span>
  Before the event
</p>
<div style="position:relative;width:100%;height:''' + str(ch) + '''px">
  <canvas id="''' + cid + '''"></canvas>
</div>
<script>
new Chart(document.getElementById("''' + cid + '''"),{
  type:"bar",
  data:{labels:''' + _j.dumps(ALL_RELS) + ''',
    datasets:[{data:''' + _j.dumps(counts) + ''',
      backgroundColor:''' + _j.dumps(colours) + ''',
      borderWidth:0,borderRadius:3}]},
  options:{indexAxis:"y",responsive:true,maintainAspectRatio:false,
    plugins:{legend:{display:false},
      tooltip:{callbacks:{label:function(c){
        return " "+c.parsed.x+" site"+(c.parsed.x!==1?"s":"");}}}},
    scales:{x:{beginAtZero:true,ticks:{stepSize:1,font:{size:12}},
        title:{display:true,text:"Number of sites",font:{size:12}}},
      y:{ticks:{font:{size:12,family:"monospace"}}}}}
});
</script>
</body></html>'''

out_dir = TTL_PATH.parent
chart_path = out_dir / 'allen_chart.html'
chart_path.write_text(chart_html, encoding='utf-8')
print(f"Chart written to: {chart_path}")
IFrame('allen_chart.html', width='100%', height=ch + 60)


Chart written to: allen_chart.html


## 9 · Visualisation B: Interactive Timeline of Qualifying Site Periods

Gantt-style timeline of all sites contemporary with or later than the *Clades Variana*.
The dashed red line marks AD 9. Use the controls to sort and filter by Allen relation.

In [9]:
import uuid
from pathlib import Path
from IPython.display import IFrame

def df_to_js(df):
    rows = []
    for _, r in df.iterrows():
        lbl = str(r['label']).replace('\\', '\\\\').replace('"', '\\"')
        rows.append(
            '{id:"' + str(r['site_id']) + '"'
            + ',label:"' + lbl + '"'
            + ',start:' + str(int(r['start']))
            + ',end:' + str(int(r['end']))
            + ',rel:"' + str(r['allen_rel']) + '"}'
        )
    return '[' + ','.join(rows) + ']'


u       = uuid.uuid4().hex
id_sort = 'sort' + u
id_filt = 'filt' + u
id_cont = 'cont' + u
id_leg  = 'leg'  + u
id_ctrl = 'ctrl' + u
n_sites = len(df_A)
sites_js = df_to_js(df_A)
row_h = 21
row_g = 4
mt, mb = 38, 36
svg_h = mt + n_sites * (row_h + row_g) + mb
iframe_h = svg_h + 100  # controls + legend

tl_html = '''<!DOCTYPE html>
<html><head><meta charset="utf-8">
<style>
  body{margin:0;font-family:sans-serif;padding:6px 4px 4px}
  #''' + id_ctrl + '''{display:flex;gap:12px;flex-wrap:wrap;align-items:center;
    padding:.3rem 0 .3rem}
  #''' + id_ctrl + ''' label{font-size:12px;color:#666}
  #''' + id_ctrl + ''' select{font-size:12px;padding:2px 6px;border-radius:4px;
    border:1px solid #ccc;background:#fff;color:#333}
  #''' + id_leg + '''{display:flex;gap:12px;flex-wrap:wrap;font-size:11px;
    color:#555;padding:.2rem 0 .3rem}
  #''' + id_leg + ''' span{display:flex;align-items:center;gap:4px}
  #''' + id_leg + ''' b{width:10px;height:10px;border-radius:2px;display:inline-block}
</style>
</head><body>
<div id="''' + id_ctrl + '''">
  <label>Sort by</label>
  <select id="''' + id_sort + '''">
    <option value="start">Start year</option>
    <option value="end">End year</option>
    <option value="allen">Allen relation</option>
    <option value="label">Site name</option>
  </select>
  <label>Filter</label>
  <select id="''' + id_filt + '''">
    <option value="all">All qualifying relations</option>
    <option value="after">after</option>
    <option value="contains">contains</option>
    <option value="meets">meets (ends at AD 9)</option>
  </select>
</div>
<div id="''' + id_leg + '''">
  <span><b style="background:#185FA5"></b>after &mdash; begins post-AD&nbsp;9</span>
  <span><b style="background:#3B6D11"></b>contains &mdash; spans AD&nbsp;9</span>
  <span><b style="background:#BA7517"></b>meets &mdash; ends at AD&nbsp;9</span>
  <span><b style="background:#993C1D;opacity:.7"></b>Clades Variana AD&nbsp;9</span>
</div>
<div id="''' + id_cont + '''"></div>
<script>
(function(){
  var D=''' + sites_js + ''';
  var COL={after:"#185FA5",contains:"#3B6D11",meets:"#BA7517"};
  var FIL={after:"#E6F1FB",contains:"#EAF3DE",meets:"#FAEEDA"};
  var se=document.getElementById("''' + id_sort + '''");
  var fe=document.getElementById("''' + id_filt + '''");
  var ce=document.getElementById("''' + id_cont + '''");
  function yr(v){return v<0?(-v)+" BC":"AD "+v;}
  function draw(){
    var s=se.value,f=fe.value;
    var d=f==="all"?D.slice():D.filter(function(x){return x.rel===f;});
    d.sort(function(a,b){
      if(s==="start")return a.start-b.start||a.label.localeCompare(b.label);
      if(s==="end")  return a.end  -b.end  ||a.label.localeCompare(b.label);
      if(s==="allen")return a.rel.localeCompare(b.rel)||a.start-b.start;
      return a.label.localeCompare(b.label);
    });
    var LW=192,RH=21,RG=4,MT=38,MB=36,CW=490,MN=-18,MX=32;
    var W=LW+CW+20,H=MT+d.length*(RH+RG)+MB;
    function px(y){return LW+(y-MN)/(MX-MN)*CW;}
    var ticks=[-15,-10,-5,0,5,9,10,15,20,25,30];
    var o="";
    o+='<svg xmlns="http://www.w3.org/2000/svg" width="'+W+'" height="'+H+'"';
    o+=' style="font-family:sans-serif;overflow:visible">';
    ticks.forEach(function(t){
      var p=px(t),ev=(t===9);
      o+='<line x1="'+p+'" y1="'+(MT-4)+'" x2="'+p+'" y2="'+(H-MB)+'"';
      o+=' stroke="'+(ev?"#993C1D":"#ddd")+'"';
      o+=' stroke-width="'+(ev?1.5:.5)+'"';
      if(ev)o+=' stroke-dasharray="4 3"';
      o+='/>';
      o+='<text x="'+p+'" y="'+(MT-8)+'" text-anchor="middle"';
      o+=' font-size="10" fill="'+(ev?"#993C1D":"#999")+'">';
      o+=(ev?"AD 9":yr(t));
      o+='</text>';
    });
    d.forEach(function(x,i){
      var y=MT+i*(RH+RG);
      var x1=px(x.start),x2=px(x.end),bw=Math.max(x2-x1,3);
      var sc=COL[x.rel]||"#888",fc=FIL[x.rel]||"#eee";
      o+='<text x="'+(LW-5)+'" y="'+(y+RH*.72)+'" text-anchor="end"';
      o+=' font-size="11" fill="#444">'+x.label+'</text>';
      o+='<rect x="'+x1+'" y="'+(y+2)+'" width="'+bw+'" height="'+(RH-4)+'"';
      o+=' rx="3" fill="'+fc+'" stroke="'+sc+'" stroke-width="1.2"/>';
      var rng=yr(x.start)+"\u2013"+yr(x.end);
      var lx=bw>68?x1+bw/2:x2+3,ta=bw>68?"middle":"start",tc=bw>68?sc:"#999";
      o+='<text x="'+lx+'" y="'+(y+RH*.72)+'" text-anchor="'+ta+'"';
      o+=' font-size="10" fill="'+tc+'">'+rng+'</text>';
    });
    var ex=px(9);
    o+='<rect x="'+(ex-40)+'" y="'+(H-MB+4)+'" width="80" height="16"';
    o+=' rx="3" fill="#FAECE7" stroke="#993C1D" stroke-width=".8"/>';
    o+='<text x="'+ex+'" y="'+(H-MB+14)+'" text-anchor="middle"';
    o+=' font-size="10" fill="#993C1D">Clades Variana</text>';
    o+='</svg>';
    ce.innerHTML=o;
  }
  se.addEventListener("change",draw);
  fe.addEventListener("change",draw);
  draw();
})();
</script>
</body></html>'''

out_dir = TTL_PATH.parent
tl_path = out_dir / 'timeline.html'
tl_path.write_text(tl_html, encoding='utf-8')
print(f"Timeline written to: {tl_path}")
IFrame('timeline.html', width='100%', height=iframe_h)


Timeline written to: timeline.html
